In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import AnyMessage,SystemMessage,HumanMessage,AIMessage,FunctionMessage,ToolMessage,BaseMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant", #llama-3.1-8b-instant,openai/gpt-oss-120b,openai/gpt-oss-20b
    temperature=0.0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

#print(llm.invoke("What is the capital of France?"))
# messages = [
#     SystemMessage(content="You are a helpful assistant.Answer in one sentence."),
#     HumanMessage(content="What is the captial of france"),
# ]
# response = llm.invoke(messages)
# print("❯❯❯❯Final response ")
# print(f"❯❯❯❯Response Type {response.type}")
# print(f"❯❯❯❯Response Content {response.content}")


In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
import operator

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [ ]:
def chat_node(state:MessagesState)-> dict:
    print("❯❯❯❯ chat_node called,Agent is responding...")
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a helpful AI assistant."
                    "Use conversation memory properly and answer based on previous response."
                )
            )

        ]
        + state["messages"]
    )
    return {
        "messages": [response.content],
        "llm-calls": state.get("llm_calls",0) + 1, 
        }
    #response = llm.invoke(state['messages'])
    #return {"messages":[response.content]}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("chat_node",chat_node)

builder.add_edge(START,"chat_node")
builder.add_edge("chat_node",END)

# Add short-term memory
memory = InMemorySaver()

graph = builder.compile(checkpointer=memory)

# Correct structure
config = {
    "configurable": {
        "thread_id": "session_123"
    }
}

In [ ]:
response = graph.invoke(
    {
        "messages": [
           HumanMessage(content="what is my name")
        ],
        "llm_calls": 0
    },
    config= config
)
#print(response)
print("❯❯❯❯ final output ")

for message in response["messages"]:
    print(message)
    